In [4]:
import pandas as pd
from datetime import datetime
import os
import glob
import re

# --- Thai month mapping ---
thai_months = {
    "ม.ค.": "Jan", "ก.พ.": "Feb", "มี.ค.": "Mar", "เม.ย.": "Apr",
    "พ.ค.": "May", "มิ.ย.": "Jun", "ก.ค.": "Jul", "ส.ค.": "Aug",
    "ก.ย.": "Sep", "ต.ค.": "Oct", "พ.ย.": "Nov", "ธ.ค.": "Dec"
}

def convert_thai_date(date_str):
    """Convert Thai/English mixed date formats or Excel datetimes to British dd/mm/yyyy."""
    if pd.isna(date_str) or str(date_str).strip() == "":
        return ""

    # --- Handle actual datetime objects from Excel directly ---
    if isinstance(date_str, (datetime, pd.Timestamp)):
        return date_str.strftime("%d/%m/%Y")

    date_str = str(date_str).strip()

    # Replace Thai month names with English abbreviations
    for thai, eng in thai_months.items():
        if thai in date_str:
            date_str = date_str.replace(thai, eng)

    # Try parsing string formats
    for fmt in ["%d %b %y", "%d %b %Y"]:
        try:
            dt = datetime.strptime(date_str, fmt)
            # Adjust Buddhist Era years
            if dt.year > datetime.now().year + 1:
                dt = dt.replace(year=dt.year - 543)
            return dt.strftime("%d/%m/%Y")
        except ValueError:
            continue

    return date_str  # fallback


def normalize_days(value):
    """Convert different text formats (1 day, 2 months, 1 year) into days (int/float)."""
    if pd.isna(value):
        return ""

    if isinstance(value, (int, float)):
        return value

    text = str(value).strip().lower()

    # Extract number
    match = re.match(r"(\d+(\.\d+)?)", text)
    if not match:
        return ""

    num = float(match.group(1))

    if "month" in text:
        return int(num * 30)
    elif "year" in text:
        return int(num * 365)
    elif "day" in text:
        return int(num)
    else:
        return num  # fallback, just keep the number
    
def clean_name(raw_name):
    """Remove Thai honorifics like นาง, นางสาว, นาย, and keep only the name."""
    if not isinstance(raw_name, str):
        return ""

    # Remove the "ของ " prefix
    name = raw_name.replace("ของ ", "").strip()

    # Common Thai honorifics to remove
    for title in ["นางสาว", "นาง", "นาย"]:
        if name.startswith(title):
            name = name.replace(title, "", 1).strip()

    return name

def transform_excel(input_file):
    """Read and transform one Excel file into the target format."""
    df_raw = pd.read_excel(input_file, header=None)

    # Full name usually in row 2
    full_name = clean_name(df_raw.iloc[1, 0])

    # Training table usually starts at row 6
    df = pd.read_excel(input_file, skiprows=6)

    df = df.rename(columns={
        df.columns[0]: "No",
        df.columns[1]: "Year",
        df.columns[2]: "Start Date",
        df.columns[3]: "End Date",
        df.columns[4]: "Days",
        df.columns[5]: "Course Name",
        df.columns[6]: "Organizing Agency",
        df.columns[7]: "Course Code"
    })

    # Build final formatted DataFrame
    output = pd.DataFrame({
        "Full Name": full_name,
        "email": "",
        "Course Name": df["Course Name"],
        "Enrolment Date": df["Start Date"].apply(convert_thai_date),
        "Completion Date": df["End Date"].apply(convert_thai_date),
        "Number of Days": df["Days"].apply(normalize_days),
        "Intake No.": "",
        "Organizing Agency": df["Organizing Agency"]
    })

    return output


def transform_folder(input_folder, output_file):
    """Process all Excel files in a folder and save combined result."""
    all_files = glob.glob(os.path.join(input_folder, "*.xlsx"))
    combined = []

    for f in all_files:
        print(f"Processing {f} ...")
        try:
            df = transform_excel(f)
            combined.append(df)
        except Exception as e:
            print(f"❌ Error processing {f}: {e}")

    if combined:
        final_df = pd.concat(combined, ignore_index=True)
        final_df.to_excel(output_file, index=False)
        print(f"✅ Combined file saved to {output_file}")
    else:
        print("⚠️ No valid Excel files processed.")


# --- Example usage ---
# transform_folder("input_excels", "all_training_combined.xlsx")

transform_folder("/Users/phonavitra/Desktop/Moodle/fwtraining", "all_training_combined.xlsx")


Processing /Users/phonavitra/Desktop/Moodle/fwtraining/รวมTraining update 19กย67.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบประวัติการฝึกอบรม_F-D3-14_นางอชิรญา  ไพรสุวรรณ.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบประวัติการฝึกอบรม_F-D3-14_นายกฤษณ์ เหลืองวัฒนพงศ์.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบประวัติการฝึกอบรม_F-D3-14_นางสาวภัทรียา ทวีกาญจน์.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบประวัติการฝึกอบรม_F-D3-14_นางปณิธิดา ศิลาเหลือง.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบฟอร์มประวัติการฝึก_F-D3-14_นางสาวปฐวีวรรณ รัตนพงศ์.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบฟอร์มประวัติการฝึก_F-D3-14_นางสาวอาฑิตา ลักษณะนุวงศ์.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบประวัติการฝึกอบรม_F-D3-14_นางสาวสุชาดา ธีระพันธุ์.xlsx ...
Processing /Users/phonavitra/Desktop/Moodle/fwtraining/แบบประวัติการฝึกอบรม_F-D3-14_นายเรวุฒ